In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src_model.config import (
    comparison_path_v2,
    feature_meta_path_v2,
    figures_dir,
    model_card_path,
    prediction_v2_path,
    report_dir
)
from src_model.training_v2 import load_model_ready_data
from src_model.testing_v2 import evaluate_on_test, evaluate_on_train, predict_test
from src_model.utils import load_json, load_model
from src_model.error_analysis import (
    build_prediction_frame,
    save_prediction_test,
    load_prediction_test,
    error_summary,
    error_by_group,
    run_error_analysis_plots,
    plot_actual_vs_predicted,
    plot_residual_distribution,
    plot_error_by_town,
    plot_error_by_flat_type,
    plot_error_by_price_band,
    plot_error_over_time
)
from src_model.interpretability import run_interpretability, extract_feature_importance
from src_model.model_card import write_model_card

sns.set_theme(style = 'whitegrid')
pd.set_option('display.float_format', '{:.2f}'.format)
print('ROOT:', ROOT)
print('figures_dir:', figures_dir)
print('report_dir:', report_dir)

ROOT: E:\AI study\HDB_resale_market_analysis
figures_dir: E:\AI study\HDB_resale_market_analysis\src_model\..\figures
report_dir: E:\AI study\HDB_resale_market_analysis\src_model\..\report


In [2]:
cmp = pd.read_csv(comparison_path_v2)
ok = cmp.dropna(subset = ['rmse']).sort_values('rmse')
display(ok)
best_name = ok.iloc[0]['model']
print('Best model:', best_name)

,rmse,mae,mape,r2,model,n_test
0,38710.90,27236.59,4.17,0.97,xgboost_v2,17507
1,41660.67,28693.42,4.36,0.96,random_forest_v2,17507
2,42895.91,30547.13,4.66,0.96,lightgbm_v2,17507
3,78161.65,54114.65,8.25,0.87,ridge_v2,17507
4,78166.78,54120.38,8.25,0.87,linear_regression_v2,17507
5,272241.60,204543.47,27.39,-0.62,median_baseline_v2,17507


Best model: xgboost_v2


In [3]:
X_train, X_test, y_train, y_test = load_model_ready_data()
print(X_train.shape, X_test.shape)
y_pred = predict_test(best_name, X_test)
pred_df = build_prediction_frame(X_test, y_test, y_pred, version = 'v2')
pred_path = save_prediction_test(pred_df, version = 'v2')
display(pred_df[['actual', 'predicted', 'residual', 'abs_error']].describe())
print(error_summary(pred_df))

(221758, 21) (17507, 21)


,actual,predicted,residual,abs_error
count,17507.00,17507.00,17507.00,17507.00
mean,661168.73,660893.01,275.72,27236.59
std,214008.65,200678.07,38711.02,27509.00
min,238888.00,271524.22,-250860.50,6.50
25%,513888.00,521606.19,-21380.22,9128.66
50%,630000.00,637517.56,-2175.56,19705.00
75%,780000.00,769848.25,17917.28,36470.59
max,1728000.00,1561200.50,434200.62,434200.62


{'n': 17507, 'rmse': 38710.895967331984, 'mae': 27236.5887854858, 'median_ae': 19705.0, 'mean_residual': 275.72096232935394, 'median_residual': -2175.5625}


In [4]:
paths = run_error_analysis_plots(pred_df, model_label = str(best_name))
paths

{'actual_vs_pred': 'E:\\AI study\\HDB_resale_market_analysis\\src_model\\..\\figures\\model_actual_vs_predicted.png',
 'residual_dist': 'E:\\AI study\\HDB_resale_market_analysis\\src_model\\..\\figures\\model_residual_distribution.png',
 'error_town': 'E:\\AI study\\HDB_resale_market_analysis\\src_model\\..\\figures\\model_error_by_subzone.png',
 'error_flat_type': 'E:\\AI study\\HDB_resale_market_analysis\\src_model\\..\\figures\\model_error_by_flat_type.png',
 'error_price_band': 'E:\\AI study\\HDB_resale_market_analysis\\src_model\\..\\figures\\model_error_by_price_band.png',
 'error_time': 'E:\\AI study\\HDB_resale_market_analysis\\src_model\\..\\figures\\model_error_over_time.png'}

In [7]:
if 'flat_type' in pred_df.columns:
    display(error_by_group(pred_df, 'flat_type', min_n = 1))
if 'town' in pred_df.columns:
    display(error_by_group(pred_df, 'town', min_n = 30).head(20))
elif 'subzone' in pred_df.columns:
    display(error_by_group(pred_df, 'subzone', min_n = 30).head(20))
if 'subzone' in pred_df.columns:
    display(error_by_group(pred_df, 'subzone', min_n = 20).head(20))
if 'nearest_mrt' in pred_df.columns:
    display(error_by_group(pred_df, 'nearest_mrt', min_n = 30).head(20))

,flat_type,n,mae,median_ae,rmse,mean_residual
6,MULTI-GENERATION,4,81203.16,70136.50,90859.30,81203.16
5,EXECUTIVE,1079,38997.17,29461.94,52401.25,2626.06
4,5 ROOM,4075,34047.29,24884.31,47779.79,2797.40
3,4 ROOM,7735,25619.36,19143.25,36260.96,-235.21
2,3 ROOM,4076,21900.09,16859.00,29469.90,-2374.88
1,2 ROOM,530,15351.89,12189.44,20515.92,3545.73
0,1 ROOM,8,14810.46,13415.11,18484.13,-13825.55


,subzone,n,mae,median_ae,rmse,mean_residual
18,OUTRAM,72,50889.72,46709.38,65267.95,25464.43
21,QUEENSTOWN,538,48451.29,38122.28,62712.60,21014.58
2,BISHAN,279,45545.11,31877.81,61458.20,-25.90
16,MARINE PARADE,87,43087.63,35732.81,53896.20,6844.80
22,ROCHOR,50,41664.02,36185.14,54588.81,5224.22
4,BUKIT MERAH,649,38576.89,28404.12,54081.50,5808.59
6,BUKIT TIMAH,43,36004.76,26619.88,46621.55,13453.62
29,TOA PAYOH,666,35875.74,28565.94,47801.34,2080.70
15,KALLANG,393,34730.85,25949.75,46267.17,9833.41
1,BEDOK,917,33268.49,20892.62,55689.44,2781.75


,subzone,n,mae,median_ae,rmse,mean_residual
18,OUTRAM,72,50889.72,46709.38,65267.95,25464.43
21,QUEENSTOWN,538,48451.29,38122.28,62712.60,21014.58
2,BISHAN,279,45545.11,31877.81,61458.20,-25.90
16,MARINE PARADE,87,43087.63,35732.81,53896.20,6844.80
22,ROCHOR,50,41664.02,36185.14,54588.81,5224.22
4,BUKIT MERAH,649,38576.89,28404.12,54081.50,5808.59
6,BUKIT TIMAH,43,36004.76,26619.88,46621.55,13453.62
29,TOA PAYOH,666,35875.74,28565.94,47801.34,2080.70
15,KALLANG,393,34730.85,25949.75,46267.17,9833.41
1,BEDOK,917,33268.49,20892.62,55689.44,2781.75


,nearest_mrt,n,mae,median_ae,rmse,mean_residual
6,BAYSHORE MRT STATION,70,122552.05,95593.47,157069.39,105183.10
105,PUNGGOL POINT LRT STATION,41,85850.35,86157.88,100080.09,83913.71
95,OUTRAM PARK MRT STATION,39,65155.01,63846.12,78038.47,36487.77
56,HAVELOCK MRT STATION,30,58906.42,38787.12,73896.53,33969.11
57,HOLLAND VILLAGE MRT STATION,50,53901.90,42433.12,74259.15,29078.46
106,QUEENSTOWN MRT STATION,295,53457.28,41957.56,68769.11,30270.64
13,BISHAN MRT STATION,187,49891.69,34459.50,66031.85,115.23
102,POTONG PASIR MRT STATION,116,45037.59,29622.97,62170.01,18132.48
89,MOUNTBATTEN MRT STATION,41,44312.42,22123.19,71327.78,39464.79
108,REDHILL MRT STATION,106,43874.87,31863.11,59180.22,6775.58


In [8]:
model = load_model(best_name)
fi_out = run_interpretability(model, top_n = 20, model_label = str(best_name))
if fi_out['importance_table'] is not None:
    display(fi_out['importance_table'].head(20))
print('FI figure:', fi_out['figure_path'])

,feature,importance
0,flat_type_EXECUTIVE,0.10
1,to_city,0.07
2,floor_area_sqm,0.06
3,flat_type_2 ROOM,0.06
4,flat_type_3 ROOM,0.05
5,flat_type_4 ROOM,0.04
6,trx_year,0.04
7,flat_type_5 ROOM,0.03
8,lease_commence,0.03
9,flat_model_Model A,0.02


FI figure: E:\AI study\HDB_resale_market_analysis\src_model\..\figures\model_feature_importance.png


In [9]:
test_metrics = evaluate_on_test(best_name, X_test, y_test)
try:
    train_metrics = evaluate_on_train(best_name, X_train, y_train)
except Exception as e:
    train_metrics = None
    print('Train metrics skipped:', e)
meta = load_json(feature_meta_path_v2)
err = error_summary(pred_df)
print('TEST:', test_metrics)
print('TRAIN:', train_metrics)
print('ERR:', err)

2026-09-15 01:38:39 | INFO     | src_model.testing_v2 | xgboost_v2 | RMSE = 38711  MAE = 27237  MAPE = 4.17%  R2 = 0.9673


TEST: {'rmse': 38710.895967331984, 'mae': 27236.5887854858, 'mape': 4.165743844931641, 'r2': 0.9672788461436387, 'model': 'xgboost_v2', 'n_test': 17507}
TRAIN: {'train_rmse': 24770.5117708482, 'train_mae': 18161.564758610737, 'train_mape': 3.6003083374600577, 'train_r2': 0.9823882901053395, 'model': 'xgboost_v2', 'n_train': 221758}
ERR: {'n': 17507, 'rmse': 38710.895967331984, 'mae': 27236.5887854858, 'median_ae': 19705.0, 'mean_residual': 275.72096232935394, 'median_residual': -2175.5625}


In [10]:
card_path = write_model_card(
    model_name = str(best_name),
    version = 'v2',
    test_metrics = test_metrics,
    train_metrics = train_metrics,
    feature_columns = meta.get('feature_columns'),
    n_train = meta.get('n_train', len(X_train)),
    n_test = meta.get('n_test', len(X_test)),
    error_summary = err,
    notes = (
        'Figures: model_actual_vs_predicted, model_residual_distribution, '
        'model_error_by_town/flat_type/price_band, model_over_time, '
        'model_feature_importance under ./figures/.'
    )
)
print('Model card written to:', card_path)

2026-09-15 01:49:12 | INFO     | src_model.model_card | Wrote model card -> E:\AI study\HDB_resale_market_analysis\src_model\..\report\model_card.md


Model card written to: E:\AI study\HDB_resale_market_analysis\src_model\..\report\model_card.md
